# QuantAlpha AI — Step 4: Sector Engine

The fundamental quality backtest (Step 3C) showed a real, broad-based edge (57% win rate, +5.28
point 12-month edge) — but 3 strong-fundamental IT stocks (TCS, INFY, HCLTECH) dragged the
average down because of IT-sector-wide weakness, not company-specific problems.

**This confirms exactly why a Sector Engine matters:** a stock can have excellent fundamentals
and still underperform if its whole sector is out of favor. This engine measures sector strength
so we can adjust confidence accordingly — matches Section 4.6 of your project doc.

**What it computes:**
1. Map each Nifty 200 stock to its sector (via yfinance's sector/industry field)
2. Compute each sector's average return (3m/6m/12m) — sector momentum
3. Score each STOCK partly on how its sector is performing, not just the company alone
4. Re-check: does adjusting for sector strength improve the quality-signal edge from Step 3C?


## 1. Connect to Drive + load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import yfinance as yf
import time
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)

technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)
print(f"Loaded {len(symbols)} stocks")


Mounted at /content/drive
Loaded 200 stocks


## 2. Fetch sector/industry mapping for each stock
Pulled from yfinance's `.info` — free, but note it can be slow (one request per stock) and
occasionally missing for some tickers.


In [2]:
def get_sector_industry(symbol):
    try:
        info = yf.Ticker(symbol + ".NS").info
        return {
            'symbol_short': symbol,
            'sector': info.get('sector', 'Unknown'),
            'industry': info.get('industry', 'Unknown')
        }
    except Exception:
        return {'symbol_short': symbol, 'sector': 'Unknown', 'industry': 'Unknown'}

sector_data = []
for i, sym in enumerate(symbols):
    sector_data.append(get_sector_industry(sym))
    if i % 25 == 0:
        print(f"Progress: {i}/{len(symbols)}")
    time.sleep(0.2)

sector_df = pd.DataFrame(sector_data)
sector_df.to_csv('data/sector_mapping.csv', index=False)
print(f"\nSector mapping done. Sectors found:")
print(sector_df['sector'].value_counts())


Progress: 0/200
Progress: 25/200
Progress: 50/200
Progress: 75/200
Progress: 100/200
Progress: 125/200
Progress: 150/200
Progress: 175/200

Sector mapping done. Sectors found:
sector
Financial Services        47
Industrials               28
Consumer Cyclical         26
Basic Materials           20
Healthcare                17
Technology                17
Consumer Defensive        15
Utilities                 11
Energy                     8
Real Estate                6
Communication Services     5
Name: count, dtype: int64


## 3. Compute each stock's historical returns (reuse from Step 3C logic)
Needed to calculate sector-level average performance.


In [3]:
def compute_forward_returns(symbol):
    path = f"data/technical/{symbol}.parquet"
    if not os.path.exists(path):
        return None
    df = pd.read_parquet(path)
    if len(df) < 260:
        return None
    df = df.reset_index(drop=True)
    current_price = df['Close'].iloc[-1]

    def price_n_days_ago(n):
        idx = len(df) - 1 - n
        return df['Close'].iloc[idx] if idx >= 0 else np.nan

    p3, p6, p12 = price_n_days_ago(63), price_n_days_ago(126), price_n_days_ago(252)
    return {
        'symbol_short': symbol,
        'return_3m': (current_price - p3) / p3 if pd.notna(p3) else np.nan,
        'return_6m': (current_price - p6) / p6 if pd.notna(p6) else np.nan,
        'return_12m': (current_price - p12) / p12 if pd.notna(p12) else np.nan,
    }

returns_data = [compute_forward_returns(sym) for sym in symbols]
returns_df = pd.DataFrame([r for r in returns_data if r])
print(f"Computed returns for {len(returns_df)} stocks")


Computed returns for 194 stocks


## 4. Compute sector-level average performance

In [4]:
master = sector_df.merge(returns_df, on='symbol_short', how='inner')
master = master.merge(fundamental_scores[['symbol_short', 'piotroski_f_score', 'roce']], on='symbol_short', how='left')

sector_performance = master.groupby('sector').agg(
    avg_return_3m=('return_3m', 'mean'),
    avg_return_6m=('return_6m', 'mean'),
    avg_return_12m=('return_12m', 'mean'),
    stock_count=('symbol_short', 'count')
).round(4)

sector_performance = sector_performance.sort_values('avg_return_12m', ascending=False)
sector_performance[['avg_return_3m', 'avg_return_6m', 'avg_return_12m']] = (
    sector_performance[['avg_return_3m', 'avg_return_6m', 'avg_return_12m']] * 100
).round(2)

print("=== Sector Performance Ranking (12-month, descending) ===\n")
print(sector_performance.to_string())
sector_performance.to_csv('data/sector_performance.csv')


=== Sector Performance Ranking (12-month, descending) ===

                        avg_return_3m  avg_return_6m  avg_return_12m  stock_count
sector                                                                           
Healthcare                      19.55          15.66           23.31           16
Utilities                       23.46          23.42           21.29           11
Communication Services          20.11          -0.43           12.49            5
Industrials                     15.86          10.06            9.21           28
Basic Materials                  3.63          -0.65            8.50           20
Financial Services              12.20          -0.89            7.44           44
Energy                           9.56          -1.35            3.60            8
Consumer Cyclical               12.91          -2.51            2.20           25
Consumer Defensive               9.80          -4.28           -1.41           15
Real Estate                     34.38  

## 5. Sector-adjusted score
Each stock now gets a `sector_momentum_score` (0-100, percentile rank of its sector's 6-month
return) alongside its existing fundamental score. A stock with great fundamentals but a weak
sector gets flagged, rather than silently blended into an average.


In [5]:
sector_return_rank = sector_performance['avg_return_6m'].rank(pct=True) * 100
master['sector_momentum_score'] = master['sector'].map(sector_return_rank).fillna(50)

# Flag: strong fundamentals but weak sector (exactly the TCS/INFY/HCLTECH pattern from Step 3C)
master['fundamentals_strong'] = master['piotroski_f_score'] >= 7
master['sector_weak'] = master['sector_momentum_score'] < 30
master['quality_trap_flag'] = master['fundamentals_strong'] & master['sector_weak']

print(f"Stocks with strong fundamentals but weak sector momentum ('quality trap' watch list): {master['quality_trap_flag'].sum()}")
master[master['quality_trap_flag']][['symbol_short', 'sector', 'piotroski_f_score', 'sector_momentum_score', 'return_12m']].sort_values('return_12m')


Stocks with strong fundamentals but weak sector momentum ('quality trap' watch list): 39


,symbol_short,sector,piotroski_f_score,sector_momentum_score,return_12m
103,KPITTECH,Technology,7,9.090909,-0.552972
180,TRENT,Consumer Cyclical,7,27.272727,-0.461916
99,JUBLFOOD,Consumer Cyclical,8,27.272727,-0.386793
174,TCS,Technology,7,9.090909,-0.371497
89,INFY,Technology,8,9.090909,-0.321864
69,HCLTECH,Technology,7,9.090909,-0.312865
100,KALYANKJIL,Consumer Cyclical,8,27.272727,-0.310663
62,GODFRYPHLP,Consumer Defensive,7,18.181818,-0.266699
42,COFORGE,Technology,9,9.090909,-0.234023
49,DIXON,Technology,8,9.090909,-0.168288


## 6. Re-test: does sector-adjusting improve the quality signal from Step 3C?
Same "High Quality" test as before, but now EXCLUDING stocks flagged as quality-trap (strong
fundamentals, weak sector) — does removing them tighten the edge?


In [6]:
master['high_quality'] = (master['piotroski_f_score'] >= 7) & (master['roce'] >= master['roce'].quantile(0.75))
master['high_quality_sector_aware'] = master['high_quality'] & (~master['quality_trap_flag'])

print("=== Original High Quality (no sector filter) ===")
for period in ['return_6m', 'return_12m']:
    hq = master[master['high_quality']][period].dropna()
    rest = master[~master['high_quality']][period].dropna()
    print(f"{period}: High Quality mean={hq.mean()*100:.2f}% (n={len(hq)}) vs Rest mean={rest.mean()*100:.2f}% | edge={(hq.mean()-rest.mean())*100:+.2f} pts")

print("\n=== Sector-Aware High Quality (excludes quality-trap stocks) ===")
for period in ['return_6m', 'return_12m']:
    hq = master[master['high_quality_sector_aware']][period].dropna()
    rest = master[~master['high_quality_sector_aware']][period].dropna()
    print(f"{period}: Sector-Aware mean={hq.mean()*100:.2f}% (n={len(hq)}) vs Rest mean={rest.mean()*100:.2f}% | edge={(hq.mean()-rest.mean())*100:+.2f} pts")


=== Original High Quality (no sector filter) ===
return_6m: High Quality mean=4.76% (n=30) vs Rest mean=1.50% | edge=+3.26 pts
return_12m: High Quality mean=10.46% (n=30) vs Rest mean=5.19% | edge=+5.28 pts

=== Sector-Aware High Quality (excludes quality-trap stocks) ===
return_6m: Sector-Aware mean=14.39% (n=16) vs Rest mean=0.89% | edge=+13.49 pts
return_12m: Sector-Aware mean=29.33% (n=16) vs Rest mean=3.90% | edge=+25.43 pts


## 7. Summary
Saved:
- `data/sector_mapping.csv` — sector/industry per stock
- `data/sector_performance.csv` — 3m/6m/12m average returns per sector, ranked
- Quality-trap watchlist (Section 5) — strong-fundamental stocks currently dragged down by weak
  sector momentum (worth monitoring for eventual recovery, but risky to buy now)

**Check the Section 6 comparison** — if the sector-aware edge is meaningfully higher than the
original, this confirms sector-awareness genuinely improves the Long-Term/Position signal, and we
build it permanently into the scoring engine.

**Next (Step 5):** Explainable AI narrative generator + entry/target/stop-loss calculation,
built on top of this now sector-aware quality signal.
